# Pathumma × Thai Words — evidence-aware semantic calibration v6

This round tests Pathumma only as a **semantic judge**. Style/register/mood/usage enrichment is intentionally removed from this prompt and will be calibrated separately later.

v6 changes after inspecting prompt-v2 errors:
- adds `definition_head` so dictionary glosses before examples are not ignored;
- detects cross-reference-only definitions such as `ดูใน หัว ๑` and forbids semantic invention;
- requires an exact `evidence_quote` copied from the candidate definition;
- cross-reference-only/missing definitions must be `unclear` with utility 0;
- keeps deterministic decoding for reproducibility;
- **does not run the 300-pair pilot yet**; this 10-pair calibration must improve first.


In [ ]:
!nvidia-smi


In [ ]:
!rm -rf /content/thai_word
!git clone --depth 1 --branch experiment/pathumma-data-enrichment-colab-2026-09-18 https://github.com/SealNM/thai_word.git /content/thai_word
%cd /content/thai_word


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

import sys, torch, transformers, bitsandbytes as bnb
print("python:", sys.version)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("bitsandbytes:", bnb.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

## Choose input

If an approved historical JSONL is already in the repo, it will be used automatically.

If not, Colab will immediately open a file picker so you can upload the approved JSONL from your computer. If you cancel or upload nothing, the notebook will fall back to a tiny technical fixture only for model-loading/JSON smoke testing.


In [ ]:
from pathlib import Path
import json

candidates = [
    Path("evaluation/writer_relevance_phase5_historical_70.approved.jsonl"),
    Path("evaluation/writer_relevance_50_annotations.approved.jsonl"),
    Path("evaluation/writer_relevance_phase4_holdout_annotations.approved.jsonl"),
]
existing = [p for p in candidates if p.exists()]
print("Historical files found:", [str(p) for p in existing])

In [ ]:
from google.colab import files

uploaded_names = []
if not existing:
    print("No approved historical JSONL found in repo.")
    print("Please choose the approved JSONL file from your computer now.")
    uploaded = files.upload()
    uploaded_names = list(uploaded)
    print("Uploaded:", uploaded_names)
else:
    print("Using historical file from repo:", existing[0])


In [ ]:
# Fallback technical fixture only if no repo file and no upload were provided.
fixture_path = Path("experiments/pathumma/pathumma_smoke_fixture.jsonl")
fixture_path.parent.mkdir(parents=True, exist_ok=True)

def smoke_row(query_id, pair_id, query, candidate, rank, utility, relation, style_tags=None):
    return {
        "schema_version": 3,
        "query_id": query_id,
        "pair_id": pair_id,
        "query": query,
        "candidate": candidate,
        "retrieval": {"v25_rank": rank},
        "annotation": {
            "utility": utility,
            "semantic_relation": relation,
            "style_tags": style_tags or [],
        },
    }

if existing:
    INPUT_PATH = str(existing[0])
    DATA_MODE = "historical_approved_repo"
elif uploaded_names:
    INPUT_PATH = uploaded_names[0]
    DATA_MODE = "historical_approved_upload"
else:
    q_fon = {
        "word": "ฝน", "sense": 1,
        "definition": "น้ำที่ตกลงมาจากเมฆเป็นเม็ดหรือหยด",
        "intended": "ฝนในความหมายของหยาดน้ำจากท้องฟ้า",
        "category": "noun:nature",
    }
    q_suay = {
        "word": "สวย", "sense": 1,
        "definition": "มีลักษณะงามน่าพึงพอใจ",
        "intended": "ความงามทางรูปลักษณ์",
        "category": "adjective:appearance",
    }
    rows = [
        smoke_row("ฝน#smoke", "smoke_fon_1", q_fon, {"word":"พิรุณ","sense":1,"definition":"ฝน; คำที่ใช้ในทางวรรณศิลป์"}, 1, 3, "direct", ["literary"]),
        smoke_row("ฝน#smoke", "smoke_fon_2", q_fon, {"word":"ฝนซู่","sense":1,"definition":"ฝนที่ตกลงมาแรงเป็นช่วงสั้น ๆ"}, 2, 2, "subtype", []),
        smoke_row("ฝน#smoke", "smoke_fon_3", q_fon, {"word":"พยับเมฆ","sense":1,"definition":"กลุ่มเมฆหรือเมฆครึ้ม"}, 3, 1, "scene_context", ["literary"]),
        smoke_row("ฝน#smoke", "smoke_fon_4", q_fon, {"word":"แสงแดด","sense":1,"definition":"แสงจากดวงอาทิตย์"}, 4, 0, "unrelated", []),
        smoke_row("ฝน#smoke", "smoke_fon_5", q_fon, {"word":"น้ำใต้ดิน","sense":1,"definition":"น้ำที่สะสมอยู่ใต้ผิวดิน"}, 5, 0, "unrelated", []),
        smoke_row("สวย#smoke", "smoke_suay_1", q_suay, {"word":"งาม","sense":1,"definition":"มีลักษณะดีน่าดู"}, 1, 3, "direct", []),
        smoke_row("สวย#smoke", "smoke_suay_2", q_suay, {"word":"งดงาม","sense":1,"definition":"งามมาก น่าชื่นชม"}, 2, 3, "direct", []),
        smoke_row("สวย#smoke", "smoke_suay_3", q_suay, {"word":"โสภา","sense":1,"definition":"งาม สวย"}, 3, 3, "direct", ["literary"]),
        smoke_row("สวย#smoke", "smoke_suay_4", q_suay, {"word":"น่าเกลียด","sense":1,"definition":"ไม่น่าดู ตรงข้ามกับสวย"}, 4, 0, "opposite_misleading", []),
        smoke_row("สวย#smoke", "smoke_suay_5", q_suay, {"word":"วิ่ง","sense":1,"definition":"เคลื่อนที่ไปโดยเร็วด้วยเท้า"}, 5, 0, "unrelated", []),
    ]
    with fixture_path.open("w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    INPUT_PATH = str(fixture_path)
    DATA_MODE = "technical_fixture"

print("Using:", INPUT_PATH)
print("Mode:", DATA_MODE)


## Run the evidence-aware 10-pair calibration

This uses prompt v3 with source-evidence checks and deterministic greedy decoding on the same 2 queries × 5 candidates used in the earlier rounds.

Do not scale to 300 pairs from this notebook. Send the report and predictions back first so we can compare v1 → v2 → v3 on exactly the same sample.


In [ ]:
import subprocess, sys
from pathlib import Path

output_dir = Path("experiments/pathumma")
for name in [
    "pathumma_predictions.jsonl",
    "pathumma_raw_generations.jsonl",
    "pathumma_report.json",
]:
    (output_dir / name).unlink(missing_ok=True)

cmd = [
    sys.executable,
    "-m", "scripts.pathumma_writer_enrichment",
    "--input", INPUT_PATH,
    "--load-in-4bit",
    "--query-limit", "2",
    "--candidates-per-query", "5",
    "--pairs-per-prompt", "5",
    "--max-new-tokens", "2048",
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
print("\nExit code:", result.returncode)

report_path = output_dir / "pathumma_report.json"
print("Report exists:", report_path.exists())

if report_path.exists():
    diagnostic = json.loads(report_path.read_text(encoding="utf-8"))
    print(json.dumps(diagnostic, ensure_ascii=False, indent=2))

if result.returncode != 0:
    raise RuntimeError(
        "Pathumma runner failed. The real error is printed above. "
        "If a diagnostic report exists, send its fatal_error field back to ChatGPT."
    )

In [ ]:
report_path = Path("experiments/pathumma/pathumma_report.json")
if not report_path.exists():
    raise RuntimeError(
        "No report was created. Re-run the previous runner cell in this v6 notebook "
        "and read the actual model/input error printed there."
    )

report = json.loads(report_path.read_text(encoding="utf-8"))
report

## Inspect disagreements

This joins predictions back to Gold **after inference**. Gold labels were never part of the Pathumma prompt.


In [ ]:
def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

gold = {r["pair_id"]: r for r in read_jsonl(INPUT_PATH)}
pred = read_jsonl("experiments/pathumma/pathumma_predictions.jsonl")

disagreements = []
for p in pred:
    g = gold.get(p["pair_id"])
    if not g:
        continue
    ga = g["annotation"]
    if ga["utility"] != p["utility"] or ga["semantic_relation"] != p["semantic_relation"]:
        disagreements.append({
            "query": g["query"]["word"],
            "candidate": g["candidate"]["word"],
            "gold_utility": ga["utility"],
            "pred_utility": p["utility"],
            "gold_relation": ga["semantic_relation"],
            "pred_relation": p["semantic_relation"],
            "confidence": p["confidence"],
            "reason": p["reason"],
        })

print("Predictions:", len(pred))
print("Disagreements:", len(disagreements))
disagreements[:30]

## Download this calibration's artifacts

Send both files back after the run. They contain the aggregate metrics and pair-level evidence-aware predictions.


In [ ]:
from google.colab import files

for artifact in [
    output_dir / "pathumma_report.json",
    output_dir / "pathumma_predictions.jsonl",
]:
    if artifact.exists():
        print("Downloading:", artifact)
        files.download(str(artifact))


### Output policy

Keep Pathumma outputs as experimental evidence only. Do not merge predictions into Gold/Silver automatically.
